In [2]:
  
from dotenv import load_dotenv , find_dotenv
load_dotenv()
import os

load_dotenv(find_dotenv())
print(os.getenv("LANGSMITH_TRACING"), os.getenv("LANGSMITH_PROJECT"))

true my-first-agent


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print(model.invoke("Explain tokens in one sentence."))

content='A token is a discrete unit of text—such as a word, sub‑word piece, or character—that a language model processes and counts when generating or interpreting language.' additional_kwargs={'reasoning_content': 'The user asks: "Explain tokens in one sentence." Need to give a concise definition. Should be one sentence. Provide answer.'} response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 77, 'total_tokens': 147, 'completion_time': 0.146752377, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.002869098, 'prompt_tokens_details': None, 'queue_time': 0.348568315, 'total_time': 0.149621475}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0c371-64b0-7291-a32f-97bd20650cb6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 70, 'total_tokens': 147, 'output_token_d

In [4]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

model_with_tools = model.bind_tools([multiply])

resp = model_with_tools.invoke("What is 47 times 89?")
print(resp.content)       # probably empty
print(resp.tool_calls)    # the model's *request* to call multiply

47 × 89 = 4183.
[]


In [5]:
from langchain.agents import create_agent

@tool
def get_word_length(word: str) -> int:
    """Return the number of characters in a word."""
    return len(word)

agent = create_agent(
    model=model,
    tools=[multiply, get_word_length],
    system_prompt="Use tools for any arithmetic or counting.",
)

out = agent.invoke(
    {"messages": [{"role": "user", "content": "How long is 'extraordinary', and what is that number times 12?"}]}
)

for m in out["messages"]:
    m.pretty_print()

================================ Human Message =================================

How long is 'extraordinary', and what is that number times 12?
================================== Ai Message ==================================
Tool Calls:
  get_word_length (fc_75b75ebf-b6be-4aad-9ce3-eb25009aa01c)
 Call ID: fc_75b75ebf-b6be-4aad-9ce3-eb25009aa01c
  Args:
    word: extraordinary
================================= Tool Message =================================
Name: get_word_length

13
================================== Ai Message ==================================
Tool Calls:
  multiply (fc_4dfb5053-163c-4200-a674-55001a380ae7)
 Call ID: fc_4dfb5053-163c-4200-a674-55001a380ae7
  Args:
    a: 13
    b: 12
================================= Tool Message =================================
Name: multiply

156
================================== Ai Message ==================================

The word “extraordinary” has **13** letters, and \(13 \times 12 = \mathbf{156}\).


In [11]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=1)

print(model.invoke("Explain tokens in one sentence.").content)

Tokens are the discrete pieces of text—ranging from whole words to sub‑word fragments or punctuation—that a language model breaks input into and processes sequentially.
